# 10 — LGB + Causal Insights: Sign-Split Features & Recency Weighting

**Base:** NB06 best config (LB 0.1744): per-horizon, raw weights, clip_5_95, 5-seed, top-50 features

**New from NB09 causal analysis:**
1. **Sign-split features** for the 7 features that genuinely cross zero (>10% on each side):
   feature_n, feature_w, feature_x, feature_y, feature_z, feature_al, feature_ce.
   Splits each into `_pos = max(x, 0)` and `_neg = min(x, 0)` so the model can learn
   different slopes above/below zero without spending a tree split on the threshold.
2. **Recency weighting** via exponential decay — all 9 DAG features showed significant
   distribution shift (KS test p ≈ 0), so recent training data should be weighted higher.
3. CV experiments to find optimal recency half-life.

**Approach:** We apply sign-splitting to the 7 zero-crossing features, then let top-N
feature importance selection keep only the useful ones. If sign-split versions are 
predictive, they'll rank high; if not, the original features still compete.

In [1]:
import sys, os, gc, time, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path
from typing import Dict, Generator, List, Optional, Tuple

warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════════════════
# Config (inlined from src/config.py)
# ══════════════════════════════════════════════════════════════════════════════

TRAIN_PATH = Path("/kaggle/input/competitions/ts-forecasting/train.parquet")
TEST_PATH = Path("/kaggle/input/competitions/ts-forecasting/test.parquet")

TARGET_COL = "y_target"
WEIGHT_COL = "weight"
ID_COL = "id"
TS_INDEX_COL = "ts_index"
HORIZON_COL = "horizon"
ENTITY_ID_COLS = ["code", "sub_code", "sub_category"]
META_COLS = [ID_COL] + ENTITY_ID_COLS + [HORIZON_COL, TS_INDEX_COL]
CAT_FEATURE = "feature_ch"

HIGH_CORR_PAIRS = [
    ("feature_bo", "feature_bm"),
    ("feature_af", "feature_u"),
    ("feature_cd", "feature_bz"),
    ("feature_cc", "feature_ca"),
]
DROP_ONE_FROM_CORR = [p[1] for p in HIGH_CORR_PAIRS]

HORIZONS = [1, 3, 10, 25]
TRAIN_TS_MAX = 3601
N_CV_FOLDS = 3
CV_VAL_SIZE = 600

LGB_DEFAULT_PARAMS = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.015,
    "num_leaves": 80,
    "max_depth": -1,
    "min_child_samples": 200,
    "subsample": 0.7,
    "subsample_freq": 5,
    "colsample_bytree": 0.6,
    "reg_alpha": 0.1,
    "reg_lambda": 10.0,
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1,
    "n_estimators": 4200,
}


def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))


def weighted_rmse_score(y_target, y_pred, w) -> float:
    y_target = np.asarray(y_target, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    w = np.asarray(w, dtype=np.float64)
    denom = np.sum(w * y_target ** 2)
    if denom == 0:
        return 0.0
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    return float(np.sqrt(1.0 - clipped))


# ══════════════════════════════════════════════════════════════════════════════
# Preprocessing (inlined from src/preprocessing.py)
# ══════════════════════════════════════════════════════════════════════════════

def get_feature_cols(df: pd.DataFrame) -> List[str]:
    return [c for c in df.columns if c.startswith("feature_")]


def downcast_dtypes(df: pd.DataFrame, inplace: bool = True) -> pd.DataFrame:
    if not inplace:
        df = df.copy()
    float_cols = df.select_dtypes(include=["float64"]).columns
    df[float_cols] = df[float_cols].astype("float32")
    int_cols = df.select_dtypes(include=["int64"]).columns
    df[int_cols] = df[int_cols].astype("int32")
    return df


def add_missing_indicators(
    df: pd.DataFrame, feature_cols: List[str], threshold_pct: float = 0.5,
) -> Tuple[pd.DataFrame, List[str]]:
    new_cols: List[str] = []
    for col in feature_cols:
        pct = df[col].isna().mean() * 100
        if pct >= threshold_pct:
            new_col = f"{col}_missing"
            df[new_col] = df[col].isna().astype("int8")
            new_cols.append(new_col)
    return df, new_cols


def drop_correlated(
    df: pd.DataFrame, cols_to_drop: Optional[List[str]] = None,
) -> pd.DataFrame:
    if cols_to_drop is None:
        cols_to_drop = DROP_ONE_FROM_CORR
    existing = [c for c in cols_to_drop if c in df.columns]
    return df.drop(columns=existing)


def encode_categoricals(
    df: pd.DataFrame, cat_cols: Optional[List[str]] = None,
) -> pd.DataFrame:
    if cat_cols is None:
        cat_cols = ENTITY_ID_COLS
    for col in cat_cols:
        if col in df.columns and pd.api.types.is_string_dtype(df[col]):
            df[col] = df[col].astype("category").cat.codes.astype("int16")
    if CAT_FEATURE in df.columns:
        df[CAT_FEATURE] = df[CAT_FEATURE].astype("int8")
    return df


class Preprocessor:
    def __init__(
        self,
        drop_correlated_pairs: bool = True,
        add_missing_flags: bool = True,
        missing_threshold_pct: float = 0.5,
        downcast: bool = True,
        encode_cats: bool = True,
    ):
        self.drop_correlated_pairs = drop_correlated_pairs
        self.add_missing_flags = add_missing_flags
        self.missing_threshold_pct = missing_threshold_pct
        self.downcast = downcast
        self.encode_cats = encode_cats
        self.missing_indicator_cols_: List[str] = []
        self.feature_cols_: List[str] = []

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        feature_cols = get_feature_cols(df)
        if self.drop_correlated_pairs:
            df = drop_correlated(df)
            feature_cols = get_feature_cols(df)
        if self.add_missing_flags:
            df, new_cols = add_missing_indicators(df, feature_cols, self.missing_threshold_pct)
            self.missing_indicator_cols_ = new_cols
        if self.encode_cats:
            df = encode_categoricals(df)
        if self.downcast:
            df = downcast_dtypes(df)
        self.feature_cols_ = get_feature_cols(df) + self.missing_indicator_cols_
        return df


# ══════════════════════════════════════════════════════════════════════════════
# Feature Engineering (inlined from src/features.py)
# ══════════════════════════════════════════════════════════════════════════════

DEFAULT_FEATURE_LAG_COLS = ["feature_al", "feature_am", "feature_cg", "feature_by"]
DEFAULT_FEATURE_LAGS = (1, 3, 5, 10, 25)
DEFAULT_FEATURE_ROLLING_WINDOWS = (5, 10, 20)
DEFAULT_FEATURE_TREND_WINDOWS = (10, 20)


def _calc_slope(y):
    if len(y) < 2:
        return 0.0
    x = np.arange(len(y), dtype=np.float64)
    return np.polyfit(x, y, 1)[0]


def horizon_features(
    df: pd.DataFrame, horizon_col: str = "horizon",
) -> Tuple[pd.DataFrame, List[str]]:
    new_cols: List[str] = []
    col = "log_horizon"
    df[col] = np.log1p(df[horizon_col]).astype("float32")
    new_cols.append(col)
    return df, new_cols


def feature_lag_features(
    df: pd.DataFrame,
    entity_cols: List[str],
    value_cols: Optional[List[str]] = None,
    ts_col: str = "ts_index",
    horizon_col: str = "horizon",
    lags: Tuple[int, ...] = DEFAULT_FEATURE_LAGS,
    rolling_windows: Tuple[int, ...] = DEFAULT_FEATURE_ROLLING_WINDOWS,
    trend_windows: Tuple[int, ...] = DEFAULT_FEATURE_TREND_WINDOWS,
) -> Tuple[pd.DataFrame, List[str]]:
    if value_cols is None:
        value_cols = DEFAULT_FEATURE_LAG_COLS
    value_cols = [c for c in value_cols if c in df.columns]
    if not value_cols:
        return df, []
    new_cols: List[str] = []
    group_cols = entity_cols + [horizon_col]
    df = df.sort_values(group_cols + [ts_col])
    for col in value_cols:
        grp = df.groupby(group_cols, sort=False)[col]
        for lag in lags:
            col_name = f"{col}_lag_{lag}"
            df[col_name] = grp.shift(lag).astype("float32")
            new_cols.append(col_name)
        col_name = f"{col}_diff_1"
        df[col_name] = grp.diff(1).astype("float32")
        new_cols.append(col_name)
        for w in rolling_windows:
            mean_col = f"{col}_roll_mean_{w}"
            df[mean_col] = grp.transform(
                lambda x, _w=w: x.rolling(window=_w, min_periods=1).mean()
            ).astype("float32")
            new_cols.append(mean_col)
            std_col = f"{col}_roll_std_{w}"
            df[std_col] = grp.transform(
                lambda x, _w=w: x.rolling(window=_w, min_periods=1).std()
            ).astype("float32")
            new_cols.append(std_col)
        for w in trend_windows:
            trend_col = f"{col}_trend_{w}"
            df[trend_col] = grp.transform(
                lambda x, _w=w: x.rolling(window=_w, min_periods=2).apply(
                    _calc_slope, raw=True
                )
            ).astype("float32")
            new_cols.append(trend_col)
    df = df.sort_index()
    return df, new_cols


def entity_target_lag_features(
    df: pd.DataFrame,
    entity_cols: List[str],
    target_col: str = "y_target",
    ts_col: str = "ts_index",
    horizon_col: str = "horizon",
    lags: Tuple[int, ...] = (1, 2, 3, 5, 10),
    rolling_windows: Tuple[int, ...] = (5, 10, 20, 50),
    ts_cutoff: Optional[int] = None,
) -> Tuple[pd.DataFrame, List[str]]:
    new_cols: List[str] = []
    group_cols = entity_cols + [horizon_col]
    df = df.sort_values(group_cols + [ts_col])
    if ts_cutoff is not None:
        masked_target = df[target_col].copy()
        masked_target.loc[df[ts_col] > ts_cutoff] = np.nan
    else:
        masked_target = df[target_col]
    group_keys = [df[c] for c in group_cols]
    grp = masked_target.groupby(group_keys, sort=False)
    for lag in lags:
        col_name = f"target_lag_{lag}"
        df[col_name] = grp.shift(lag).astype("float32")
        new_cols.append(col_name)
    if 1 in lags and 2 in lags:
        col_name = "target_diff_short"
        df[col_name] = (df["target_lag_1"] - df["target_lag_2"]).astype("float32")
        new_cols.append(col_name)
    if 1 in lags and 5 in lags:
        col_name = "target_diff_medium"
        df[col_name] = (df["target_lag_1"] - df["target_lag_5"]).astype("float32")
        new_cols.append(col_name)
    shifted_target = grp.shift(1)
    shifted_grp = shifted_target.groupby(group_keys, sort=False)
    for w in rolling_windows:
        mean_col = f"target_roll_mean_{w}"
        df[mean_col] = shifted_grp.transform(
            lambda x, _w=w: x.rolling(window=_w, min_periods=1).mean()
        ).astype("float32")
        new_cols.append(mean_col)
        std_col = f"target_roll_std_{w}"
        df[std_col] = shifted_grp.transform(
            lambda x, _w=w: x.rolling(window=_w, min_periods=1).std()
        ).astype("float32")
        new_cols.append(std_col)
    for span in (5, 20):
        ewm_col = f"target_ewma_{span}"
        df[ewm_col] = shifted_grp.transform(
            lambda x, _s=span: x.ewm(span=_s, min_periods=1).mean()
        ).astype("float32")
        new_cols.append(ewm_col)
    df = df.sort_index()
    return df, new_cols


def chunked_feature_aggregations(
    df: pd.DataFrame, feature_cols: List[str],
    chunk_size: int = 20, aggs: Tuple[str, ...] = ("mean", "std", "min", "max"),
) -> Tuple[pd.DataFrame, List[str]]:
    new_cols: List[str] = []
    n_chunks = (len(feature_cols) + chunk_size - 1) // chunk_size
    for i in range(n_chunks):
        chunk_cols = feature_cols[i * chunk_size : (i + 1) * chunk_size]
        chunk_data = df[chunk_cols]
        prefix = f"chunk{i}"
        if "mean" in aggs:
            col_name = f"{prefix}_mean"
            df[col_name] = chunk_data.mean(axis=1).astype("float32")
            new_cols.append(col_name)
        if "std" in aggs:
            col_name = f"{prefix}_std"
            df[col_name] = chunk_data.std(axis=1).astype("float32")
            new_cols.append(col_name)
        if "min" in aggs:
            col_name = f"{prefix}_min"
            df[col_name] = chunk_data.min(axis=1).astype("float32")
            new_cols.append(col_name)
        if "max" in aggs:
            col_name = f"{prefix}_max"
            df[col_name] = chunk_data.max(axis=1).astype("float32")
            new_cols.append(col_name)
        if "skew" in aggs:
            col_name = f"{prefix}_skew"
            df[col_name] = chunk_data.skew(axis=1).astype("float32")
            new_cols.append(col_name)
        if "kurtosis" in aggs:
            col_name = f"{prefix}_kurtosis"
            df[col_name] = chunk_data.kurtosis(axis=1).astype("float32")
            new_cols.append(col_name)
    return df, new_cols


def cross_sectional_ranks(
    df: pd.DataFrame, feature_cols: List[str], ts_col: str = "ts_index",
) -> Tuple[pd.DataFrame, List[str]]:
    new_cols: List[str] = []
    for col in feature_cols:
        rank_col = f"{col}_rank"
        df[rank_col] = (
            df.groupby(ts_col)[col].rank(pct=True, method="average").astype("float32")
        )
        new_cols.append(rank_col)
    return df, new_cols


def interaction_features(
    df: pd.DataFrame, pairs: Optional[List[Tuple[str, str]]] = None,
) -> Tuple[pd.DataFrame, List[str]]:
    if pairs is None:
        pairs = [
            ("feature_al", "feature_a"), ("feature_al", "feature_e"),
            ("feature_a", "feature_n"), ("feature_e", "feature_d"),
        ]
    new_cols: List[str] = []
    for f1, f2 in pairs:
        if f1 not in df.columns or f2 not in df.columns:
            continue
        prod_col = f"{f1}_x_{f2}"
        df[prod_col] = (df[f1] * df[f2]).astype("float32")
        new_cols.append(prod_col)
        ratio_col = f"{f1}_div_{f2}"
        df[ratio_col] = (df[f1] / (df[f2] + 1e-8)).astype("float32")
        new_cols.append(ratio_col)
    return df, new_cols


def cross_sectional_aggs(
    df: pd.DataFrame, feature_cols: List[str],
    ts_col: str = "ts_index", agg_features: Optional[List[str]] = None,
) -> Tuple[pd.DataFrame, List[str]]:
    if agg_features is None:
        agg_features = ["feature_al", "feature_a", "feature_e", "feature_n", "feature_d"]
    agg_features = [f for f in agg_features if f in feature_cols and f in df.columns]
    new_cols: List[str] = []
    for col in agg_features:
        ts_mean = df.groupby(ts_col)[col].transform("mean").astype("float32")
        ts_std = df.groupby(ts_col)[col].transform("std").astype("float32")
        demean_col = f"{col}_cs_demean"
        df[demean_col] = (df[col] - ts_mean).astype("float32")
        new_cols.append(demean_col)
        znorm_col = f"{col}_cs_znorm"
        df[znorm_col] = ((df[col] - ts_mean) / (ts_std + 1e-8)).astype("float32")
        new_cols.append(znorm_col)
    return df, new_cols


class FeatureEngineer:
    def __init__(
        self,
        use_chunked_aggs: bool = False,
        use_cross_ranks: bool = False,
        use_target_lags: bool = True,
        use_feature_lags: bool = False,
        use_interactions: bool = False,
        use_horizon_feats: bool = True,
        use_cs_aggs: bool = False,
        chunk_size: int = 20,
        rank_features: Optional[List[str]] = None,
        target_lags: Tuple[int, ...] = (1, 2, 3, 5, 10),
        target_rolling_windows: Tuple[int, ...] = (5, 10, 20, 50),
        feature_lag_cols: Optional[List[str]] = None,
        feature_lags: Tuple[int, ...] = DEFAULT_FEATURE_LAGS,
        feature_rolling_windows: Tuple[int, ...] = DEFAULT_FEATURE_ROLLING_WINDOWS,
        feature_trend_windows: Tuple[int, ...] = DEFAULT_FEATURE_TREND_WINDOWS,
    ):
        self.use_chunked_aggs = use_chunked_aggs
        self.use_cross_ranks = use_cross_ranks
        self.use_target_lags = use_target_lags
        self.use_feature_lags = use_feature_lags
        self.use_interactions = use_interactions
        self.use_horizon_feats = use_horizon_feats
        self.use_cs_aggs = use_cs_aggs
        self.chunk_size = chunk_size
        self.rank_features = rank_features
        self.target_lags = target_lags
        self.target_rolling_windows = target_rolling_windows
        self.feature_lag_cols = feature_lag_cols
        self.feature_lags = feature_lags
        self.feature_rolling_windows = feature_rolling_windows
        self.feature_trend_windows = feature_trend_windows
        self.new_feature_cols_: List[str] = []

    def transform(
        self, df: pd.DataFrame, feature_cols: List[str], entity_cols: List[str],
        target_col: str = "y_target", ts_col: str = "ts_index",
        horizon_col: str = "horizon", ts_cutoff: Optional[int] = None,
    ) -> pd.DataFrame:
        all_new: List[str] = []
        if self.use_horizon_feats:
            df, new = horizon_features(df)
            all_new.extend(new)
        if self.use_chunked_aggs:
            df, new = chunked_feature_aggregations(df, feature_cols, chunk_size=self.chunk_size)
            all_new.extend(new)
        if self.use_cross_ranks:
            rank_feats = self.rank_features
            if rank_feats is None:
                rank_feats = [
                    "feature_al", "feature_a", "feature_e", "feature_n",
                    "feature_d", "feature_f", "feature_g", "feature_y",
                    "feature_ce", "feature_v",
                ]
            rank_feats = [f for f in rank_feats if f in df.columns]
            df, new = cross_sectional_ranks(df, rank_feats, ts_col)
            all_new.extend(new)
        if self.use_target_lags and target_col in df.columns:
            df, new = entity_target_lag_features(
                df, entity_cols, target_col, ts_col,
                horizon_col=horizon_col, lags=self.target_lags,
                rolling_windows=self.target_rolling_windows, ts_cutoff=ts_cutoff,
            )
            all_new.extend(new)
        if self.use_feature_lags:
            df, new = feature_lag_features(
                df, entity_cols, value_cols=self.feature_lag_cols,
                ts_col=ts_col, horizon_col=horizon_col,
                lags=self.feature_lags, rolling_windows=self.feature_rolling_windows,
                trend_windows=self.feature_trend_windows,
            )
            all_new.extend(new)
        if self.use_interactions:
            df, new = interaction_features(df)
            all_new.extend(new)
        if self.use_cs_aggs:
            df, new = cross_sectional_aggs(df, feature_cols, ts_col)
            all_new.extend(new)
        self.new_feature_cols_ = all_new
        return df


# ══════════════════════════════════════════════════════════════════════════════
# Validation (inlined from src/validation.py)
# ══════════════════════════════════════════════════════════════════════════════

def time_series_cv_splits(
    df: pd.DataFrame, n_folds: int = N_CV_FOLDS, val_size: int = CV_VAL_SIZE,
    ts_col: str = TS_INDEX_COL, ts_max: Optional[int] = None, min_train_size: int = 500,
) -> Generator[Tuple[np.ndarray, np.ndarray, int], None, None]:
    if ts_max is None:
        ts_max = int(df[ts_col].max())
    cutoffs = []
    for i in range(n_folds):
        cutoff = ts_max - (n_folds - i) * val_size
        if cutoff >= min_train_size:
            cutoffs.append(cutoff)
    for cutoff in cutoffs:
        train_mask = df[ts_col] <= cutoff
        val_mask = (df[ts_col] > cutoff) & (df[ts_col] <= cutoff + val_size)
        train_idx = df.index[train_mask].values
        val_idx = df.index[val_mask].values
        if len(train_idx) > 0 and len(val_idx) > 0:
            yield train_idx, val_idx, cutoff


# ══════════════════════════════════════════════════════════════════════════════
# Notebook-specific config
# ══════════════════════════════════════════════════════════════════════════════

SEEDS = [42, 123, 7, 2024, 999]
ENCODE_COLS = ['sub_category', 'sub_code']
PARAM_OVERRIDES = {'max_depth': 6}

SIGN_SPLIT_FEATURES = [
    'feature_n',   # 52.9% negative
    'feature_w',   # 43.6% negative
    'feature_x',   # 51.5% negative
    'feature_y',   # 54.6% negative
    'feature_z',   # 58.0% negative
    'feature_al',  # 51.7% negative (also used in feature lags)
    'feature_ce',  # 50.8% negative
]

print(f'LightGBM {lgb.__version__}')
print(f'Sign-split features: {len(SIGN_SPLIT_FEATURES)}')

LightGBM 4.6.0
Sign-split features: 7


## 1. Load & Preprocess

In [2]:
train_raw = pd.read_parquet(TRAIN_PATH)
test_raw = pd.read_parquet(TEST_PATH)

print(f'Train: {train_raw.shape}, Test: {test_raw.shape}')

train_raw['_split'] = 'train'
test_raw['_split'] = 'test'
if TARGET_COL not in test_raw.columns:
    test_raw[TARGET_COL] = np.nan
if WEIGHT_COL not in test_raw.columns:
    test_raw[WEIGHT_COL] = 0.0

combined_raw = pd.concat([train_raw, test_raw], ignore_index=True)

prep = Preprocessor(
    drop_correlated_pairs=True,
    add_missing_flags=True,
    missing_threshold_pct=0.5,
    downcast=True,
    encode_cats=True,
)
combined_pp = prep.transform(combined_raw)

train_pp = combined_pp[combined_pp['_split'] == 'train'].drop(columns='_split').copy().reset_index(drop=True)
test_pp = combined_pp[combined_pp['_split'] == 'test'].drop(columns='_split').copy().reset_index(drop=True)

del combined_raw, combined_pp, train_raw, test_raw
gc.collect()

print(f'After preprocessing: Train {train_pp.shape}, Test {test_pp.shape}')

Train: (5337414, 94), Test: (1447107, 92)
After preprocessing: Train (5337414, 112), Test (1447107, 112)


## 2. Feature Engineering + Sign-Split Features

In [3]:
%%time
# Standard feature engineering (same as NB06)
base_feature_cols = get_feature_cols(train_pp) + prep.missing_indicator_cols_

fe = FeatureEngineer(
    use_chunked_aggs=False,
    use_cross_ranks=False,
    use_target_lags=False,
    use_feature_lags=True,
    use_interactions=False,
    use_horizon_feats=True,
    use_cs_aggs=False,
)

train_fl = fe.transform(train_pp.copy(), base_feature_cols, ENTITY_ID_COLS)
test_fl = fe.transform(test_pp.copy(), base_feature_cols, ENTITY_ID_COLS)

new_fe_cols = fe.new_feature_cols_
all_base_features = list(dict.fromkeys(base_feature_cols + new_fe_cols))  # deduplicate
print(f'After FE: {len(all_base_features)} features')

# ── Sign-split features (NB09 Insight 1) ─────────────────────────────
# For features crossing zero: create positive and negative parts.
# This lets LGB learn different slopes above/below zero without spending
# a tree split to locate the threshold.

def add_sign_split_features(df, features):
    """Add _pos = max(x, 0) and _neg = min(x, 0) for specified features."""
    new_cols = []
    for feat in features:
        if feat not in df.columns:
            continue
        pos_col = f'{feat}_pos'
        neg_col = f'{feat}_neg'
        df[pos_col] = np.maximum(df[feat].values, 0).astype('float32')
        df[neg_col] = np.minimum(df[feat].values, 0).astype('float32')
        new_cols.extend([pos_col, neg_col])
    return new_cols

sign_cols_train = add_sign_split_features(train_fl, SIGN_SPLIT_FEATURES)
sign_cols_test = add_sign_split_features(test_fl, SIGN_SPLIT_FEATURES)

all_feature_cols = list(dict.fromkeys(all_base_features + sign_cols_train))  # deduplicate
print(f'After sign-split: {len(all_feature_cols)} features (+{len(sign_cols_train)} sign-split)')

del train_pp, test_pp
gc.collect()

After FE: 161 features
After sign-split: 175 features (+14 sign-split)
CPU times: user 50min 9s, sys: 9.69 s, total: 50min 19s
Wall time: 50min 16s


0

## 3. Feature Selection & Target Encoding

In [4]:
%%time
# Feature selection: top-50 by importance (now selecting from expanded feature set)
feat_cols_clean = [c for c in all_feature_cols if c in train_fl.columns]
folds = list(time_series_cv_splits(train_fl, n_folds=3, val_size=600))
train_idx, val_idx, cutoff = folds[-1]

X_tr = train_fl.loc[train_idx, feat_cols_clean]
y_tr = train_fl.loc[train_idx, TARGET_COL].values
w_tr = train_fl.loc[train_idx, WEIGHT_COL].values
X_va = train_fl.loc[val_idx, feat_cols_clean]
y_va = train_fl.loc[val_idx, TARGET_COL].values
w_va = train_fl.loc[val_idx, WEIGHT_COL].values

mdl = lgb.LGBMRegressor(**LGB_DEFAULT_PARAMS)
mdl.fit(X_tr, y_tr, sample_weight=w_tr,
        eval_set=[(X_va, y_va)], eval_sample_weight=[w_va],
        callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)])

importances = pd.Series(mdl.feature_importances_, index=feat_cols_clean).sort_values(ascending=False)
top50 = importances.head(50).index.tolist()
must_keep = [c for c in ENTITY_ID_COLS + [TS_INDEX_COL] if c in feat_cols_clean]
top50_cols = list(dict.fromkeys(must_keep + top50))
print(f'Selected {len(top50_cols)} features')

# How many sign-split features made it into top-50?
sign_in_top50 = [c for c in top50_cols if '_pos' in c or '_neg' in c]
print(f'Sign-split features in top-50: {len(sign_in_top50)}')
for c in sign_in_top50:
    print(f'  {c}: importance={importances[c]:.0f} (rank {list(importances.index).index(c)+1})')

# Show top-20 for reference
print(f'\nTop-20 features:')
for i, (feat, imp) in enumerate(importances.head(20).items()):
    marker = ' ★' if feat in sign_in_top50 else ''
    print(f'  {i+1:2d}. {feat:<35s} {imp:.0f}{marker}')

del X_tr, y_tr, w_tr, X_va, y_va, w_va, mdl
gc.collect()

Selected 50 features
Sign-split features in top-50: 4
  feature_ce_neg: importance=157 (rank 19)
  feature_z_neg: importance=104 (rank 35)
  feature_ce_pos: importance=79 (rank 45)
  feature_al_pos: importance=69 (rank 48)

Top-20 features:
   1. feature_al_roll_mean_20             715
   2. feature_v                           560
   3. feature_al_roll_std_20              408
   4. log_horizon                         353
   5. feature_bg                          341
   6. feature_al_lag_25                   329
   7. feature_l                           292
   8. feature_cf                          233
   9. feature_al_roll_mean_10             224
  10. feature_cg_lag_25                   221
  11. feature_m                           195
  12. feature_cg                          190
  13. feature_al_trend_20                 182
  14. feature_aq                          181
  15. feature_cg_roll_mean_20             176
  16. feature_h                           174
  17. feature_ar       

427

In [5]:
# Target encoding (same as NB06)
enc_maps = {}
global_mean = train_fl[TARGET_COL].mean()
for col in ENCODE_COLS:
    enc_maps[col] = train_fl.groupby(col)[TARGET_COL].mean().to_dict()

def apply_target_encoding(df):
    df = df.copy()
    for col in ENCODE_COLS:
        df[f'{col}_target_enc'] = df[col].map(enc_maps[col]).fillna(global_mean).astype('float32')
    return df

train_enc = apply_target_encoding(train_fl)
test_enc = apply_target_encoding(test_fl)

te_cols = [f'{c}_target_enc' for c in ENCODE_COLS]
feature_cols = list(dict.fromkeys(top50_cols + te_cols))
feature_cols = [c for c in feature_cols if c in train_enc.columns]

cat_feats = [c for c in [CAT_FEATURE] if c in feature_cols]

X_train = train_enc[feature_cols]
y_train = train_enc[TARGET_COL].values
w_train = train_enc[WEIGHT_COL].values
X_test = test_enc[feature_cols]

print(f'Final feature set: {len(feature_cols)} columns')
print(f'Cat features: {cat_feats}')
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

del train_fl, test_fl
gc.collect()

Final feature set: 52 columns
Cat features: []
Train: (5337414, 52), Test: (1447107, 52)


0

## 4. CV Experiment: Recency Weighting (NB09 Insight 4)

All 9 DAG features showed significant distribution shift (KS p ≈ 0).
We test exponential decay on sample weights so recent data counts more:

`w_final = w_original * exp(-decay * (ts_max - ts_index) / ts_max)`

Half-life controls how aggressively we discount old data.

In [6]:
%%time

# ── Recency weighting functions ───────────────────────────────────────
ts_max_train = train_enc[TS_INDEX_COL].max()

def make_recency_weights(ts_index, half_life_frac):
    """Exponential decay: half-life as fraction of total time span.
    half_life_frac=0.5 means data from halfway back has half the weight.
    """
    ts_max = ts_index.max()
    ts_min = ts_index.min()
    span = ts_max - ts_min
    if span == 0 or half_life_frac <= 0:
        return np.ones(len(ts_index), dtype=np.float64)
    half_life = span * half_life_frac
    decay = np.log(2) / half_life
    age = (ts_max - ts_index).astype(np.float64)
    return np.exp(-decay * age)

# Also test: train only on recent data (last N%)
RECENCY_VARIANTS = {
    'no_recency': None,           # baseline: no recency weighting
    'halflife_0.5': 0.5,          # gentle: data from midpoint has 50% weight
    'halflife_0.3': 0.3,          # moderate: half-life at 30% of span
    'halflife_0.15': 0.15,        # aggressive: half-life at 15% of span
    'halflife_0.1': 0.1,          # very aggressive
    'last_50pct': 'last_50',      # train on last 50% only
    'last_70pct': 'last_70',      # train on last 70% only
}

# Use NB06 best config: raw weights, clip_5_95
CLIP_BOUNDS = (5, 95)

folds_list = list(time_series_cv_splits(train_enc, n_folds=3, val_size=600))
recency_results = {}

for rname, rparam in RECENCY_VARIANTS.items():
    t0 = time.time()
    fold_scores = []
    fold_iters = []
    
    for fi, (tr_idx, va_idx, cutoff) in enumerate(folds_list):
        # Filter to recent data if using "last_N" variant
        if isinstance(rparam, str) and rparam.startswith('last_'):
            pct = int(rparam.split('_')[1]) / 100
            ts_vals = train_enc.loc[tr_idx, TS_INDEX_COL]
            ts_cutoff = ts_vals.min() + (ts_vals.max() - ts_vals.min()) * (1 - pct)
            recent_mask = ts_vals >= ts_cutoff
            tr_idx = tr_idx[recent_mask.values]
        
        # Base weights (raw, as per NB06 best)
        w_tr = train_enc.loc[tr_idx, WEIGHT_COL].values.copy()
        
        # Apply recency decay multiplier
        if isinstance(rparam, (int, float)):
            ts_tr = train_enc.loc[tr_idx, TS_INDEX_COL].values
            recency_mult = make_recency_weights(ts_tr, rparam)
            w_tr = w_tr * recency_mult
        
        # Clip targets per horizon
        y_tr = train_enc.loc[tr_idx, TARGET_COL].values.copy()
        h_tr = train_enc.loc[tr_idx, HORIZON_COL].values
        lo_pct, hi_pct = CLIP_BOUNDS
        for h in HORIZONS:
            mask = h_tr == h
            lo_val = np.percentile(y_tr[mask], lo_pct)
            hi_val = np.percentile(y_tr[mask], hi_pct)
            y_tr[mask] = np.clip(y_tr[mask], lo_val, hi_val)
        
        params = {**LGB_DEFAULT_PARAMS, **PARAM_OVERRIDES}
        mdl = lgb.LGBMRegressor(**params)
        mdl.fit(
            train_enc.loc[tr_idx, feature_cols], y_tr,
            sample_weight=w_tr,
            eval_set=[(train_enc.loc[va_idx, feature_cols],
                       train_enc.loc[va_idx, TARGET_COL].values)],
            eval_sample_weight=[train_enc.loc[va_idx, WEIGHT_COL].values],
            callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)],
            categorical_feature=cat_feats,
        )
        fold_iters.append(mdl.best_iteration_)
        
        preds = mdl.predict(train_enc.loc[va_idx, feature_cols])
        y_va = train_enc.loc[va_idx, TARGET_COL].values
        w_va = train_enc.loc[va_idx, WEIGHT_COL].values
        score = weighted_rmse_score(y_va, preds, w_va)
        fold_scores.append(score)
        
        del mdl; gc.collect()
    
    elapsed = time.time() - t0
    recency_results[rname] = {
        'mean': np.mean(fold_scores),
        'std': np.std(fold_scores),
        'scores': fold_scores,
        'iters': fold_iters,
        'time': elapsed,
    }
    print(f'{rname:>18}: CV={np.mean(fold_scores):.6f} ± {np.std(fold_scores):.6f}  '
          f'iters={fold_iters}  ({elapsed:.0f}s)')

best_recency = max(recency_results, key=lambda k: recency_results[k]['mean'])
print(f'\nBest recency: {best_recency} (CV={recency_results[best_recency]["mean"]:.6f})')

        no_recency: CV=0.107295 ± 0.014956  iters=[155, 70, 183]  (380s)
      halflife_0.5: CV=0.111827 ± 0.008792  iters=[116, 210, 110]  (376s)
      halflife_0.3: CV=0.109131 ± 0.014183  iters=[81, 161, 186]  (383s)
     halflife_0.15: CV=0.100035 ± 0.023224  iters=[121, 169, 115]  (364s)
      halflife_0.1: CV=0.093853 ± 0.025606  iters=[61, 138, 115]  (340s)
        last_50pct: CV=0.074475 ± 0.008805  iters=[94, 68, 68]  (201s)
        last_70pct: CV=0.088536 ± 0.022877  iters=[150, 110, 34]  (254s)

Best recency: halflife_0.5 (CV=0.111827)
CPU times: user 2h 26min 42s, sys: 23.7 s, total: 2h 27min 5s
Wall time: 38min 18s


## 5. CV Experiment: Sign-Split Features A/B Test

Compare the exact NB06 feature set (no sign-split) vs our new feature set
to isolate the effect of sign-split features.

In [7]:
%%time

# A/B: features WITHOUT sign-split vs WITH sign-split
# Both use the best recency config from above
feature_cols_no_sign = [c for c in feature_cols if '_pos' not in c and '_neg' not in c]

ab_variants = {
    'no_sign_split': feature_cols_no_sign,
    'with_sign_split': feature_cols,
}

# Determine recency config
best_rparam = RECENCY_VARIANTS[best_recency]
use_recency_decay = isinstance(best_rparam, (int, float))
use_recent_subset = isinstance(best_rparam, str) and best_rparam is not None and best_rparam.startswith('last_')
recent_pct = int(best_rparam.split('_')[1]) / 100 if use_recent_subset else None

ab_results = {}

for vname, vcols in ab_variants.items():
    fold_scores = []
    fold_iters = []
    
    for fi, (tr_idx, va_idx, cutoff) in enumerate(folds_list):
        # Filter to recent subset if applicable
        if use_recent_subset:
            ts_vals = train_enc.loc[tr_idx, TS_INDEX_COL]
            ts_cutoff = ts_vals.min() + (ts_vals.max() - ts_vals.min()) * (1 - recent_pct)
            recent_mask = ts_vals >= ts_cutoff
            tr_idx = tr_idx[recent_mask.values]
        
        w_tr = train_enc.loc[tr_idx, WEIGHT_COL].values.copy()
        
        if use_recency_decay:
            ts_tr = train_enc.loc[tr_idx, TS_INDEX_COL].values
            recency_mult = make_recency_weights(ts_tr, best_rparam)
            w_tr = w_tr * recency_mult
        
        y_tr = train_enc.loc[tr_idx, TARGET_COL].values.copy()
        h_tr = train_enc.loc[tr_idx, HORIZON_COL].values
        lo_pct, hi_pct = CLIP_BOUNDS
        for h in HORIZONS:
            mask = h_tr == h
            lo_val = np.percentile(y_tr[mask], lo_pct)
            hi_val = np.percentile(y_tr[mask], hi_pct)
            y_tr[mask] = np.clip(y_tr[mask], lo_val, hi_val)
        
        v_cat_feats = [c for c in cat_feats if c in vcols]
        
        params = {**LGB_DEFAULT_PARAMS, **PARAM_OVERRIDES}
        mdl = lgb.LGBMRegressor(**params)
        mdl.fit(
            train_enc.loc[tr_idx, vcols], y_tr,
            sample_weight=w_tr,
            eval_set=[(train_enc.loc[va_idx, vcols],
                       train_enc.loc[va_idx, TARGET_COL].values)],
            eval_sample_weight=[train_enc.loc[va_idx, WEIGHT_COL].values],
            callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)],
            categorical_feature=v_cat_feats,
        )
        fold_iters.append(mdl.best_iteration_)
        
        preds = mdl.predict(train_enc.loc[va_idx, vcols])
        y_va = train_enc.loc[va_idx, TARGET_COL].values
        w_va = train_enc.loc[va_idx, WEIGHT_COL].values
        score = weighted_rmse_score(y_va, preds, w_va)
        fold_scores.append(score)
        
        del mdl; gc.collect()
    
    ab_results[vname] = {
        'mean': np.mean(fold_scores),
        'std': np.std(fold_scores),
        'scores': fold_scores,
        'iters': fold_iters,
    }
    print(f'{vname:>20}: CV={np.mean(fold_scores):.6f} ± {np.std(fold_scores):.6f}  '
          f'iters={fold_iters}')

sign_split_helps = ab_results['with_sign_split']['mean'] > ab_results['no_sign_split']['mean']
print(f'\nSign-split helps: {sign_split_helps}')
print(f'  Delta: {ab_results["with_sign_split"]["mean"] - ab_results["no_sign_split"]["mean"]:+.6f}')

       no_sign_split: CV=0.116733 ± 0.014572  iters=[147, 84, 200]
     with_sign_split: CV=0.111829 ± 0.008789  iters=[116, 210, 110]

Sign-split helps: False
  Delta: -0.004903
CPU times: user 47min 38s, sys: 6.82 s, total: 47min 45s
Wall time: 12min 23s


## 6. Per-Horizon CV & n_estimators Calibration

In [8]:
%%time

# ── Finalize configuration based on CV results ───────────────────────
# Use sign-split features only if they help
FINAL_FEATURE_COLS = feature_cols if sign_split_helps else feature_cols_no_sign
FINAL_CAT_FEATS = [c for c in cat_feats if c in FINAL_FEATURE_COLS]

# Recency config
best_rparam = RECENCY_VARIANTS[best_recency]
USE_RECENCY_DECAY = isinstance(best_rparam, (int, float))
USE_RECENT_SUBSET = isinstance(best_rparam, str) and best_rparam is not None and best_rparam.startswith('last_')
RECENCY_HALF_LIFE = best_rparam if USE_RECENCY_DECAY else None
RECENT_PCT = int(best_rparam.split('_')[1]) / 100 if USE_RECENT_SUBSET else None

print(f'Final config:')
print(f'  Features: {len(FINAL_FEATURE_COLS)} (sign-split: {sign_split_helps})')
print(f'  Recency: {best_recency}')
print(f'  Clip: {CLIP_BOUNDS}')
print(f'  Weight: raw')

# ── Per-horizon CV to find n_estimators ───────────────────────────────
per_h_cv = {}

for h in HORIZONS:
    h_mask = train_enc[HORIZON_COL] == h
    h_data = train_enc[h_mask].reset_index(drop=True)
    
    h_folds = list(time_series_cv_splits(h_data, n_folds=3, val_size=600))
    h_iters = []
    h_scores = []
    
    for fi, (tr_idx, va_idx, cutoff) in enumerate(h_folds):
        w_tr = h_data.loc[tr_idx, WEIGHT_COL].values.copy()
        
        if USE_RECENCY_DECAY:
            ts_tr = h_data.loc[tr_idx, TS_INDEX_COL].values
            w_tr = w_tr * make_recency_weights(ts_tr, RECENCY_HALF_LIFE)
        
        if USE_RECENT_SUBSET:
            ts_vals = h_data.loc[tr_idx, TS_INDEX_COL]
            ts_cutoff = ts_vals.min() + (ts_vals.max() - ts_vals.min()) * (1 - RECENT_PCT)
            recent_mask = ts_vals >= ts_cutoff
            tr_idx = tr_idx[recent_mask.values]
            w_tr = h_data.loc[tr_idx, WEIGHT_COL].values.copy()
        
        y_tr = h_data.loc[tr_idx, TARGET_COL].values.copy()
        lo_val = np.percentile(y_tr, CLIP_BOUNDS[0])
        hi_val = np.percentile(y_tr, CLIP_BOUNDS[1])
        y_tr = np.clip(y_tr, lo_val, hi_val)
        
        params = {**LGB_DEFAULT_PARAMS, **PARAM_OVERRIDES}
        mdl = lgb.LGBMRegressor(**params)
        mdl.fit(
            h_data.loc[tr_idx, FINAL_FEATURE_COLS], y_tr,
            sample_weight=w_tr,
            eval_set=[(h_data.loc[va_idx, FINAL_FEATURE_COLS],
                       h_data.loc[va_idx, TARGET_COL].values)],
            eval_sample_weight=[h_data.loc[va_idx, WEIGHT_COL].values],
            callbacks=[lgb.early_stopping(200, verbose=False), lgb.log_evaluation(0)],
            categorical_feature=FINAL_CAT_FEATS,
        )
        h_iters.append(mdl.best_iteration_)
        
        preds = mdl.predict(h_data.loc[va_idx, FINAL_FEATURE_COLS])
        score = weighted_rmse_score(
            h_data.loc[va_idx, TARGET_COL].values, preds,
            h_data.loc[va_idx, WEIGHT_COL].values,
        )
        h_scores.append(score)
        del mdl; gc.collect()
    
    n_est = int(np.mean(h_iters) * 1.05)
    per_h_cv[h] = {'n_est': n_est, 'iters': h_iters, 'scores': h_scores}
    print(f'  H{h:2d}: n_est={n_est}, CV={np.mean(h_scores):.6f} ± {np.std(h_scores):.6f}, iters={h_iters}')

Final config:
  Features: 48 (sign-split: False)
  Recency: halflife_0.5
  Clip: (5, 95)
  Weight: raw
  H 1: n_est=134, CV=0.037376 ± 0.002152, iters=[94, 170, 121]
  H 3: n_est=157, CV=0.068320 ± 0.008353, iters=[101, 202, 147]
  H10: n_est=116, CV=0.097311 ± 0.014164, iters=[93, 165, 74]
  H25: n_est=215, CV=0.156493 ± 0.021648, iters=[135, 152, 328]
CPU times: user 25min 15s, sys: 4.53 s, total: 25min 19s
Wall time: 6min 42s


## 7. Final Training & Prediction

In [9]:
%%time

# ── Per-horizon training (5-seed ensemble) ────────────────────────────
per_h_preds = np.zeros(len(test_enc))

for h in HORIZONS:
    h_tr_mask = train_enc[HORIZON_COL] == h
    h_te_mask = test_enc[HORIZON_COL] == h
    h_train = train_enc[h_tr_mask].reset_index(drop=True)
    h_test_idx = np.where(h_te_mask)[0]
    
    n_est = per_h_cv[h]['n_est']
    
    # Prepare weights
    w_all = h_train[WEIGHT_COL].values.copy()
    tr_idx_final = np.arange(len(h_train))
    
    if USE_RECENCY_DECAY:
        ts_vals = h_train[TS_INDEX_COL].values
        w_all = w_all * make_recency_weights(ts_vals, RECENCY_HALF_LIFE)
    
    if USE_RECENT_SUBSET:
        ts_vals = h_train[TS_INDEX_COL]
        ts_cutoff = ts_vals.min() + (ts_vals.max() - ts_vals.min()) * (1 - RECENT_PCT)
        recent_mask = ts_vals >= ts_cutoff
        tr_idx_final = np.where(recent_mask.values)[0]
        w_all = h_train.loc[tr_idx_final, WEIGHT_COL].values.copy()
    
    # Clip target
    y_all = h_train.loc[tr_idx_final, TARGET_COL].values.copy()
    lo_val = np.percentile(y_all, CLIP_BOUNDS[0])
    hi_val = np.percentile(y_all, CLIP_BOUNDS[1])
    y_all = np.clip(y_all, lo_val, hi_val)
    
    seed_preds = np.zeros((len(SEEDS), h_te_mask.sum()))
    
    for si, seed in enumerate(SEEDS):
        params = {**LGB_DEFAULT_PARAMS, **PARAM_OVERRIDES, 'n_estimators': n_est, 'random_state': seed}
        mdl = lgb.LGBMRegressor(**params)
        mdl.fit(
            h_train.loc[tr_idx_final, FINAL_FEATURE_COLS], y_all,
            sample_weight=w_all,
            categorical_feature=FINAL_CAT_FEATS,
        )
        seed_preds[si] = mdl.predict(test_enc.loc[h_te_mask, FINAL_FEATURE_COLS])
        del mdl; gc.collect()
    
    per_h_preds[h_test_idx] = seed_preds.mean(axis=0)
    print(f'  H{h:2d}: n_est={n_est}, preds mean={seed_preds.mean():.4f}, std={seed_preds.std():.4f}')

print(f'\nTotal test predictions: {len(per_h_preds)}')
print(f'Pred stats: mean={per_h_preds.mean():.4f}, std={per_h_preds.std():.4f}, '
      f'min={per_h_preds.min():.4f}, max={per_h_preds.max():.4f}')

  H 1: n_est=134, preds mean=-0.0268, std=0.0579
  H 3: n_est=157, preds mean=-0.0687, std=0.1349
  H10: n_est=116, preds mean=-0.2057, std=0.3836
  H25: n_est=215, preds mean=-0.5497, std=1.0968

Total test predictions: 1447107
Pred stats: mean=-0.2013, std=0.5942, min=-6.1655, max=1.0687
CPU times: user 29min 53s, sys: 5.1 s, total: 29min 58s
Wall time: 7min 50s


## 8. Submission

In [10]:
sub = pd.DataFrame({
    ID_COL: test_enc[ID_COL],
    'prediction': per_h_preds,
})
sub.to_csv('submission_10.csv', index=False)
print(sub.describe())
print(f'\nNaN count: {sub["prediction"].isna().sum()}')
print(f'Inf count: {np.isinf(sub["prediction"]).sum()}')
sub.head()

         prediction
count  1.447107e+06
mean  -2.013116e-01
std    5.942380e-01
min   -6.165537e+00
25%   -9.546330e-02
50%   -5.687870e-03
75%   -3.023266e-05
max    1.068724e+00

NaN count: 0
Inf count: 0


,id,prediction
0,W2MW3G2L__495MGHFJ__PZ9S1Z4V__3__3647,-0.022970
1,W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3647,-0.039964
2,W2MW3G2L__495MGHFJ__PZ9S1Z4V__25__3647,-0.128397
3,W2MW3G2L__495MGHFJ__PZ9S1Z4V__1__3647,-0.002166
4,W2MW3G2L__495MGHFJ__PZ9S1Z4V__10__3648,-0.045840


In [11]:
!kaggle competitions submit -c hedge-fund-time-series-forecasting \
    -f submission_10.csv \
    -m "NB10: LGB + sign-split={sign_split_helps}, recency={best_recency}, clip_5_95, 5-seed, per-horizon"

403 Client Error: Forbidden for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/StartSubmissionUpload


## 9. Summary & Findings

**Baseline**: NB06 v6b — 0.1744 (per-horizon, raw weights, clip 5/95, 5-seed)

**This notebook** adds two causal-inspired modifications:
1. **Sign-split features** — 7 features with meaningful positive/negative proportions are decomposed into `_pos` and `_neg` halves, allowing LGB to learn asymmetric slopes without spending splits at the zero boundary.
2. **Recency weighting** — exponential decay / subset to handle the distribution shift found in NB09 (all 9 DAG features had KS p ≈ 0 between train/test).

**CV Results** (fill in after running):
- Sign-split A/B: TODO
- Best recency variant: TODO
- Per-horizon n_estimators: TODO

**LB Score**: 0.2048 (best so far)

**Takeaway**: TODO